# **SAGE: Практика**

Всем привет. В уроке «SHAP для разного: взаимодействия и глобальность» мы перенесли дележ Шепли с одного прогноза на качество модели в целом и получили SAGE (Shapley Additive Global importancE, [Covert, Lundberg и Lee, 2020](https://arxiv.org/abs/2004.00668)). Игроки в этой игре — признаки, а ценность коалиции $S$ — насколько знание признаков из $S$ снижает ошибку:

$$v(S) = E\big[\ell(f_\varnothing, Y)\big] - E_{X,Y}\big[\ell(f_S(X), Y)\big],$$

где $f_S$ — прогноз модели, в котором недостающие признаки маргинализованы по фону, а $f_\varnothing = E[f(X)]$ — «пустая» модель, которая всегда выдает средний прогноз модели. Важность признака $\phi_i$ — его значение Шепли в игре $(N, v)$, и по эффективности $\sum_i \phi_i = v(N)$ — полное снижение ошибки.

Сегодня посчитаем SAGE библиотекой [`sage-importance`](https://github.com/iancovert/sage) на уже знакомом наборе diabetes — на нем в том же уроке мы строили рейтинг по mean|φ|.

В этой практике вы:
- посчитаете SAGE-значения случайного леса через `sage.MarginalImputer` и `sage.PermutationEstimator`;
- проверите эффективность: сложите SAGE-значения и сравните сумму с $v(N)$, посчитанным вручную;
- сравните SAGE с перестановочной важностью из `sklearn`;
- добавите в данные почти точную копию признака и посмотрите, как SAGE и перестановочная важность делят между дублями вклад.

**Приятного кодинга!**

Как всегда, сначала установим зависимости. На Colab нет только `sage-importance`; версию закрепим, чтобы числа совпадали с нашими.

In [ ]:
!pip install sage-importance==0.0.6 -q

In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sage

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_squared_error, r2_score

## Данные и модель

Набор diabetes встроен в `sklearn`, скачивать ничего не нужно: 442 пациентки и пациента, 10 признаков (возраст, пол, индекс массы тела `bmi`, давление `bp` и шесть показателей крови `s1`–`s6`), цель — количественная мера прогрессии диабета через год. Признаки уже отцентрированы и отмасштабированы. Отложим 30% строк на тест: на тесте будем считать и SAGE, и перестановочную важность.

In [ ]:
X, y = load_diabetes(return_X_y=True, as_frame=True)
feature_names = list(X.columns)

X_train, X_test, y_train, y_test = train_test_split(X.values, y.values, test_size=0.3, random_state=0)
print(X.shape, X_train.shape, X_test.shape)
X.head()

Модель — случайный лес из 300 деревьев. Параметр `max_features=0.33` значит, что в каждом разбиении дерево выбирает лучший признак из случайной трети признаков. Для первых разделов это лишь регуляризация, а в эксперименте с дублем это свойство станет важным: лес будет пользоваться обеими копиями признака.

In [ ]:
def fit_forest(X_train, y_train):
    return RandomForestRegressor(n_estimators=300, max_features=0.33, min_samples_leaf=3,
                                 random_state=0, n_jobs=-1).fit(X_train, y_train)

model = fit_forest(X_train, y_train)
pred_test = model.predict(X_test)
print(f'MSE на тесте: {mean_squared_error(y_test, pred_test):.1f}, R^2 на тесте: {r2_score(y_test, pred_test):.3f}')

## SAGE-значения

В `sage-importance` расчет собирается из двух частей:

- **`sage.MarginalImputer(model, background)`** реализует $f_S$: недостающие признаки заменяются значениями из каждой строки фона, и прогнозы усредняются. Фон — 64 случайные строки обучающей выборки; чем он больше, тем точнее маргинализация и тем медленнее расчет.
- **`sage.PermutationEstimator(imputer, 'mse')`** оценивает значения Шепли методом Монте-Карло. Для одной случайной тестовой строки он берет случайную перестановку признаков, добавляет признаки по одному и записывает, на сколько упала квадратичная ошибка после добавления каждого признака. Среднее таких приращений по многим перестановкам и есть оценка $\phi_i$. Есть и `sage.KernelEstimator`, который решает ту же задачу через взвешенную регрессию, как KernelSHAP.

По умолчанию оценщик останавливается сам, когда доверительные интервалы становятся достаточно узкими, и тогда число перестановок меняется от запуска к запуску. Мы зафиксируем число перестановок (8192), сид оценщика и фон, чтобы при повторном запуске получались те же числа. Расчет занимает около минуты.

In [ ]:
N_PERMUTATIONS = 8192

def sage_values(model, X_train, X_test, y_test):
    random.seed(0)
    np.random.seed(0)
    background = X_train[np.random.default_rng(0).choice(len(X_train), 64, replace=False)]
    imputer = sage.MarginalImputer(model, background)                        # f_S: маргинализация по фону
    estimator = sage.PermutationEstimator(imputer, 'mse', random_state=0)    # значения Шепли перестановками
    explanation = estimator(X_test, y_test, batch_size=512, n_permutations=N_PERMUTATIONS,
                            detect_convergence=False, bar=False)
    return explanation, background

sage_expl, background = sage_values(model, X_train, X_test, y_test)
# std — стандартная ошибка оценки Монте-Карло, а не разброс по объектам
pd.DataFrame({'SAGE': sage_expl.values, 'std': sage_expl.std}, index=feature_names).sort_values('SAGE', ascending=False).round(1)

SAGE-значения измеряются в тех же единицах, что и функция потерь: $\phi_i$ — это часть снижения MSE на тесте, приходящаяся на признак $i$. У библиотеки есть встроенный график с 95% доверительными интервалами оценки.

In [ ]:
sage_expl.plot(feature_names, title='SAGE-значения, случайный лес на diabetes', figsize=(9, 5))

**Вопрос для самопроверки.** Какие два признака получили наибольшие SAGE-значения? Различает ли их эта оценка, если учесть 95% доверительные интервалы? Признаки с отрицательными SAGE-значениями — что значит минус в терминах ошибки?

<details><summary>Проверить себя</summary>

Лидируют `bmi` (648.7, стандартная ошибка 30.7) и `s5` (639.8, стандартная ошибка 28.8). Разница между ними 8.9, а половина ширины 95% доверительного интервала у каждого около 60 (1.96 стандартной ошибки), поэтому интервалы перекрываются: эта оценка `bmi` и `s5` не различает, и порядок между ними может смениться при другом сиде оценщика. Уверенно можно сказать только, что оба признака важнее `bp` (248.6). Отрицательные значения у `age` (−11.4) и `s1` (−9.8): в среднем по коалициям знание этих признаков увеличивает квадратичную ошибку на тесте, то есть модель, вероятно, подстроилась под шум этих признаков на обучающей выборке.

</details>

## Задание 1. Эффективность: сумма SAGE и $v(N)$

По теории $\sum_i \phi_i = v(N)$, где $v(N)$ — снижение ошибки моделью со всеми признаками относительно «пустой» модели. Посчитайте $v(N)$ на тесте вручную двумя способами:

1. «пустая» модель предсказывает $\bar y$ — среднее цели на обучающей выборке: так проще всего посчитать базовую ошибку;
2. как в определении из теории и как это делает `MarginalImputer`: при пустой коалиции все признаки берутся из фона, поэтому «пустой» прогноз $f_\varnothing$ — средний прогноз модели по строкам фона `background`.

Сравните обе величины с суммой SAGE-значений.

Подсказка о том, почему совпадение не точное. В одной перестановке приращения ошибки складываются в разность потерь пустой и полной модели на одной строке (промежуточные слагаемые сокращаются). Поэтому сумма оценок равна среднему этой разности по 8192 строкам, которые оценщик вытянул из теста с возвращением. Посчитайте стандартную ошибку такого среднего: стандартное отклонение разности по строкам теста, деленное на $\sqrt{8192}$.

In [ ]:
loss_full = # Ваш код здесь: средняя квадратичная ошибка полной модели на тесте

v_N_ybar = # Ваш код здесь
f_empty = # Ваш код здесь
v_N_empty = # Ваш код здесь

standard_error = # Ваш код здесь

print(f'Сумма SAGE-значений:            {sage_expl.values.sum():.1f}')
print(f'v(N) с пустой моделью y_bar:    {v_N_ybar:.1f}')
print(f'v(N) с пустой моделью f_empty:  {v_N_empty:.1f}')
print(f'Разница суммы и v(N) с f_empty: {sage_expl.values.sum() - v_N_empty:.1f}, стандартная ошибка: {standard_error:.1f}')

**Вопрос для самопроверки.** Чему равны сумма SAGE-значений и $v(N)$ на тесте при двух выборах «пустой» модели? Укладывается ли разница в две стандартные ошибки?

<details><summary>Проверить себя</summary>

Сумма SAGE-значений 1790.7. $v(N)$ с пустой моделью $\bar y = 152.1$ равно 1767.6, с пустой моделью $f_\varnothing = 148.4$ (средний прогноз по фону) — 1781.8. Разница суммы и $v(N)$ с $f_\varnothing$ равна 8.9 при стандартной ошибке 69.0, то есть укладывается в одну стандартную ошибку: эффективность выполняется с точностью до шума выборки строк. Сравнивать нужно именно с $f_\varnothing$: библиотека отсчитывает снижение ошибки от среднего прогноза по фону, а не от среднего цели.

</details>

## Задание 2. Сравнение с перестановочной важностью

Перестановочная важность перемешивает один признак при всех остальных на месте и смотрит, на сколько выросла ошибка. В терминах игры это предельный вклад признака в коалицию из всех остальных признаков, то есть одно слагаемое из суммы Шепли вместо усреднения по всем коалициям.

Посчитайте перестановочную важность случайного леса на тесте функцией `sklearn.inspection.permutation_importance` с `scoring='neg_mean_squared_error'` (тогда важность измеряется в тех же единицах MSE, что и SAGE), `n_repeats=10` и `random_state=0`. Сохраните результат в `perm`, соберите таблицу `comparison` со столбцами `SAGE` и `permutation` и сравните суммы столбцов с $v(N)$.

In [ ]:
random.seed(0)
np.random.seed(0)
perm = # Ваш код здесь
comparison = # Ваш код здесь
print(comparison.round(1))
print(f'\nСумма SAGE: {comparison.SAGE.sum():.1f}, сумма перестановочной важности: {comparison.permutation.sum():.1f}, v(N) с f_empty: {v_N_empty:.1f}')

Нарисуем обе важности рядом. Единицы у них общие — снижение MSE на тесте, поэтому столбики можно сравнивать напрямую.

In [ ]:
def plot_two(table, title):
    pos = np.arange(len(table))
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.barh(pos - 0.2, table.SAGE, height=0.4, label='SAGE', color='tab:green')
    ax.barh(pos + 0.2, table.permutation, height=0.4, label='перестановочная важность', color='tab:gray')
    ax.set_yticks(pos, table.index)
    ax.invert_yaxis()
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_xlabel('снижение MSE на тесте')
    ax.set_title(title)
    ax.legend()
    plt.show()

plot_two(comparison, 'SAGE и перестановочная важность, случайный лес на diabetes')

**Вопрос для самопроверки.** Совпадают ли две первые позиции рейтинга у SAGE и у перестановочной важности? Чему равна сумма перестановочной важности по всем признакам и почему она не обязана совпадать с $v(N)$?

<details><summary>Проверить себя</summary>

Первые две позиции совпадают: `bmi` и `s5` у обоих методов (перестановочная важность 743.9 и 734.1). Сумма перестановочной важности 1663.1 против $v(N) = 1781.8$: перестановочная важность — не значения Шепли, свойства эффективности у нее нет, и ее сумма ничего не обязана давать. Расхождения сосредоточены в показателях крови: у `s4` и `s3` SAGE-значения положительные (97.4 и 74.6), а перестановочная важность отрицательная (−27.9 и −42.8). Эти признаки коррелированы с другими (`s3` и `s4`: −0.74, `s4` и `s2`: 0.66), и когда перемешивают один, остальные остаются на месте и восполняют информацию — перестановочная важность видит только этот последний предельный вклад.

</details>

## Задание 3. Избыточный признак: как SAGE и перестановочная важность делят вклад

Теперь проверим главное обещание из теории: SAGE корректнее обходится с избыточными признаками. Добавим в данные шумную копию `s5`: `s5_dup = s5 + шум`, где стандартное отклонение шума — 10% стандартного отклонения `s5`. Новой информации о цели у `s5_dup` нет. Обучим тот же лес на 11 признаках: при `max_features=0.33` в части разбиений `s5` не попадает в случайную треть признаков, а `s5_dup` попадает, поэтому лес пользуется обеими копиями.

In [ ]:
X_dup = X.copy()
X_dup['s5_dup'] = X['s5'] + np.random.default_rng(0).normal(0, 0.1 * X['s5'].std(), len(X))
feature_names_dup = list(X_dup.columns)

X_dup_train, X_dup_test, _, _ = train_test_split(X_dup.values, y.values, test_size=0.3, random_state=0)
model_dup = fit_forest(X_dup_train, y_train)
print(f"Корреляция s5 и s5_dup: {np.corrcoef(X_dup['s5'], X_dup['s5_dup'])[0, 1]:.3f}")
print(f'MSE на тесте: без дубля {mean_squared_error(y_test, model.predict(X_test)):.1f}, с дублем {mean_squared_error(y_test, model_dup.predict(X_dup_test)):.1f}')

Посчитайте для `model_dup` SAGE-значения (функцией `sage_values` выше) и перестановочную важность (с теми же параметрами, что в задании 2). Соберите таблицу `comparison_dup` со столбцами `SAGE` и `permutation` по 11 признакам и посчитайте долю `dup_share` — какую часть суммарного SAGE-значения пары `s5` и `s5_dup` получил дубль.

In [ ]:
sage_dup, _ = # Ваш код здесь
random.seed(0)
np.random.seed(0)
perm_dup = # Ваш код здесь
comparison_dup = # Ваш код здесь
dup_share = # Ваш код здесь

pair = pd.DataFrame({
    'SAGE без дубля': [comparison.SAGE['s5'], np.nan, comparison.SAGE['s5']],
    'SAGE с дублем': [comparison_dup.SAGE['s5'], comparison_dup.SAGE['s5_dup'], comparison_dup.SAGE[['s5', 's5_dup']].sum()],
    'перестановочная без дубля': [comparison.permutation['s5'], np.nan, comparison.permutation['s5']],
    'перестановочная с дублем': [comparison_dup.permutation['s5'], comparison_dup.permutation['s5_dup'], comparison_dup.permutation[['s5', 's5_dup']].sum()],
}, index=['s5', 's5_dup', 's5 + s5_dup'])
print(pair.round(1).to_string())
print(f'\nДоля дубля в SAGE пары: {dup_share:.3f}')
print(f'Сумма SAGE: {comparison_dup.SAGE.sum():.1f}, стандартная ошибка s5 и s5_dup: {sage_dup.std[feature_names_dup.index("s5")]:.1f} и {sage_dup.std[-1]:.1f}')

Посмотрим на обе важности модели с дублем.

In [ ]:
plot_two(comparison_dup, 'Модель с дублем s5_dup: SAGE и перестановочная важность')

**Вопрос для самопроверки.** Какую долю SAGE-значения пары `s5` + `s5_dup` получил дубль? Сравните сумму SAGE-значений пары с SAGE-значением `s5` в модели без дубля. Во сколько раз упала перестановочная важность `s5` после добавления дубля и чему равна перестановочная важность пары?

<details><summary>Проверить себя</summary>

Дубль получил долю 0.489 SAGE-значения пары: `s5` 357.2, `s5_dup` 342.5, вместе 699.7 — близко к 639.8, которые `s5` получал один в модели без дубля (разница 59.9 при стандартных ошибках 20.1 и 18.8 у слагаемых, и это уже другая обученная модель). SAGE поделил вклад между копиями почти поровну и сохранил их общий вклад. Перестановочная важность `s5` упала с 734.1 до 59.9, примерно в 12 раз, у `s5_dup` — 37.8, у пары вместе 97.7: перемешивая одну копию, мы оставляем вторую на месте, лес продолжает по ней прогнозировать, и обе копии выглядят почти бесполезными.

</details>

## Заключение

Сегодня вы:
- посчитали SAGE-значения случайного леса библиотекой `sage-importance` и увидели, что у оценки Монте-Карло есть стандартная ошибка, которую нужно учитывать при сравнении признаков;
- проверили эффективность: сумма SAGE-значений оценивает $v(N)$, полное снижение ошибки относительно «пустой» модели, и важно, какой прогноз считается «пустым»;
- сравнили SAGE с перестановочной важностью: у двух лидеров рейтинги совпадают, у коррелированных показателей крови `s3` и `s4` они расходятся даже в знаке, а сумма перестановочной важности не обязана давать $v(N)$;
- на избыточном признаке увидели главное различие: SAGE делит вклад между дублями и сохраняет их общий вклад, а перестановочная важность обесценивает обе копии, потому что каждую оценивает при второй на месте.

До встречи на следующих шагах!